In [1]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.input.loaders.dfs import (
    store_entity_semantic_embeddings,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [52]:
# 步骤 1：找到排序最大的文件夹
output_path = "/home/ljc/data/graphrag/alltest/0411/cyber_v3_tobeuse_ori_only1_baseline_b/output/"
folders = [os.path.join(output_path, d) for d in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, d))]
latest_folder = max(folders, key=os.path.getmtime)
print(f"Latest folder: {latest_folder}")

Latest folder: /home/ljc/data/graphrag/alltest/0411/cyber_v3_tobeuse_ori_only1_baseline_b/output/20250421-134626


In [53]:
INPUT_DIR = latest_folder + "/artifacts"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"
RELATIONSHIP_TABLE = "create_final_relationships"
COVARIATE_TABLE = "create_final_covariates"
TEXT_UNIT_TABLE = "create_final_text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [54]:
# read nodes table to get community and degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# load description embeddings to an in-memory lancedb vectorstore
# to connect to a remote db, specify url and port values.
description_embedding_store = LanceDBVectorStore(
    collection_name="entity_description_embeddings",
)
description_embedding_store.connect(db_uri=LANCEDB_URI)
entity_description_embeddings = store_entity_semantic_embeddings(
    entities=entities, vectorstore=description_embedding_store
)

print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 5640


,level,title,type,description,source_id,community,degree,human_readable_id,id,size,graph_embedding,entity_type,top_level_node_id,x,y
0,0,P.A.S. WEBSHELL,EVENT,P.A.S. Webshell is a multifunctional PHP websh...,"0951c82da0391a8ba9474b82cfc39eee,5351a0c12cfdc...",0,7,0,b45241d70f0e43fca764df95b2b81f77,7.0,"[0.00703090988099575, -0.10878964513540268, 0....",None,b45241d70f0e43fca764df95b2b81f77,11.315439,4.571320
1,0,NETWORK SEGMENTATION,None,Network Segmentation is a cybersecurity strate...,"070f2edcfa873eafa5d870374d1bfd82,102972f6ad945...",2,35,1,4119fd06010c494caa07f439b333f4c5,35.0,"[0.09216798096895218, -0.08225885778665543, 0....",None,4119fd06010c494caa07f439b333f4c5,9.181621,1.912120
2,0,EPIC,ORGANIZATION,EPIC is a backdoor software utilized by the Tu...,"20e6c5f49c5a682c2e71314af4dab817,66e21cf04f235...",15,5,2,d3835bf3dda84ead99deadbeac5d0d7d,5.0,"[-0.0873030573129654, -0.014025380834937096, 0...",None,d3835bf3dda84ead99deadbeac5d0d7d,14.893945,9.419086
3,0,APPLICATION ISOLATION,None,Application Isolation is a security measure de...,"1338f70def968f4183c1a7ac8476b964,20e6c5f49c5a6...",15,7,3,077d2820ae1845bcbb1803379a3d1eae,7.0,"[-0.037924546748399734, -0.01435079611837864, ...",None,077d2820ae1845bcbb1803379a3d1eae,14.621139,9.741131
4,0,SANDBOXING,None,Sandboxing is a security approach that establi...,"1338f70def968f4183c1a7ac8476b964,22d0399495027...",15,6,4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,6.0,"[-0.017374888062477112, -0.004354625474661589,...",EVENT,3671ea0dd4e84c1a9b02c5ab2c8f4bac,14.633510,9.604239


In [55]:
entity_embedding_df

,id,name,type,description,human_readable_id,graph_embedding,text_unit_ids,description_embedding
0,b45241d70f0e43fca764df95b2b81f77,P.A.S. WEBSHELL,EVENT,P.A.S. Webshell is a multifunctional PHP websh...,0,"[0.00703090988099575, -0.10878964513540268, 0....","[0951c82da0391a8ba9474b82cfc39eee, 5351a0c12cf...","[-0.00929004605859518, 0.007910400629043579, 0..."
1,4119fd06010c494caa07f439b333f4c5,NETWORK SEGMENTATION,None,Network Segmentation is a cybersecurity strate...,1,"[0.09216798096895218, -0.08225885778665543, 0....","[070f2edcfa873eafa5d870374d1bfd82, 102972f6ad9...","[-0.011196316219866276, 0.00015380315016955137..."
2,d3835bf3dda84ead99deadbeac5d0d7d,EPIC,ORGANIZATION,EPIC is a backdoor software utilized by the Tu...,2,"[-0.0873030573129654, -0.014025380834937096, 0...","[20e6c5f49c5a682c2e71314af4dab817, 66e21cf04f2...","[-0.016108687967061996, -0.002848197938874364,..."
3,077d2820ae1845bcbb1803379a3d1eae,APPLICATION ISOLATION,None,Application Isolation is a security measure de...,3,"[-0.037924546748399734, -0.01435079611837864, ...","[1338f70def968f4183c1a7ac8476b964, 20e6c5f49c5...","[-0.033553238958120346, -0.0024539621081203222..."
4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,SANDBOXING,None,Sandboxing is a security approach that establi...,4,"[-0.017374888062477112, -0.004354625474661589,...","[1338f70def968f4183c1a7ac8476b964, 22d03994950...","[-0.0019393287366256118, 0.009077049791812897,..."
...,...,...,...,...,...,...,...,...
230,1fee51d6f4614127a3e1cc80d018506e,SYSTEM ISOLATION PRACTICES,EVENT,System isolation practices are methods impleme...,935,"[-0.04255814105272293, -0.05564694479107857, -...",[e03d15da9adeb311f3b3dc61d609fbc0],"[-0.02616552636027336, 0.0039058036636561155, ..."
231,00dc2c0748214e52bc799ca3e25204e9,REGISTRY SETTINGS,EVENT,Registry settings are system configurations th...,936,None,[bb1f71e4de00b9eb5174f785a8311916],"[0.022288070991635323, 0.039369016885757446, 0..."
232,42d1a9e749ad40daa34c7b0b695f8751,SECURE OPERATIONS,EVENT,Secure operations involve implementing strateg...,937,"[0.018965084105730057, -0.05776099115610123, -...",[0d6952755ce296ad02024081a779633b],"[0.03431377187371254, 0.02142193540930748, 0.0..."
233,20de9a1af6ab4e88acf003cb7be0217c,DATA TRANSITIONS,EVENT,Data transitions refer to the movement of data...,938,None,[445a73129f875b6a2a32b7e2084ea16d],"[0.018197331577539444, 0.009590825997292995, 0..."


In [56]:
entity_embedding_df.to_csv("/home/ljc/data/graphrag/alltest/0411/cyber_v3_tobeuse_ori_only1_baseline_b/entity.csv",index = False)

#### Read relationships

In [57]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 1649


,source,target,weight,description,text_unit_ids,id,human_readable_id,source_degree,target_degree,rank
0,P.A.S. WEBSHELL,NETWORK SEGMENTATION,5.0,P.A.S. Webshell is a significant cybersecurity...,"[8d23d5b42375ccfe493c1205def03e5d, 902b38128aa...",5154b4a4f3ac43729703c69fccb54633,0,7,35,42
1,P.A.S. WEBSHELL,NETWORK INTRUSION DETECTION SYSTEM,5.0,The P.A.S. Webshell is a malicious entity that...,"[0951c82da0391a8ba9474b82cfc39eee, 5351a0c12cf...",2091070e709e45f5ae56d40a9da45520,1,7,8,15
2,P.A.S. WEBSHELL,CHINA CHOPPER,9.0,Both P.A.S. Webshell and China Chopper are web...,[bd4946ea4a1b47f4b3bad655c8915fdb],09045ef5c4314dde9a631a206274563f,2,7,1,8
3,P.A.S. WEBSHELL,LINUX,6.0,"P.A.S. Webshell operates on Linux platforms, i...",[56ede4abbe4e59ebdcd15115a139143a],1b9baa98ede84164883e8cdcbc7000c1,3,7,35,42
4,P.A.S. WEBSHELL,WINDOWS,6.0,"P.A.S. Webshell operates on Windows platforms,...",[56ede4abbe4e59ebdcd15115a139143a],e4f3fcc475a74756925b730caffcb70d,4,7,55,62


In [58]:
# covariate_df = pd.read_parquet(f"{INPUT_DIR}/{COVARIATE_TABLE}.parquet")

# claims = read_indexer_covariates(covariate_df)

# print(f"Claim records: {len(claims)}")
# covariates = {"claims": claims}

#### Read community reports

In [59]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 213


,community,full_content,level,rank,title,rank_explanation,summary,findings,full_content_json,id
0,211,# Windows Operating System and Cyber Threats\n...,5,8.5,Windows Operating System and Cyber Threats,The impact severity rating is high due to the ...,The community centers around the Windows opera...,[{'explanation': 'Windows is a critical operat...,"{\n ""title"": ""Windows Operating System and ...",0e6fcb78-da95-43e5-a92d-5da1ae1c1330
1,212,# LSASS and Procdump in Windows Security\n\nTh...,5,7.5,LSASS and Procdump in Windows Security,The impact severity rating is high due to the ...,The community focuses on the interaction betwe...,[{'explanation': 'LSASS (Local Security Author...,"{\n ""title"": ""LSASS and Procdump in Windows...",4ef9a5a4-684d-4774-821f-421e755f69e4
2,205,# BoomBox Malware and Lateral Movement Threats...,4,8.5,BoomBox Malware and Lateral Movement Threats,The impact severity rating is high due to the ...,"The community focuses on the BoomBox malware, ...",[{'explanation': 'BoomBox is identified as a d...,"{\n ""title"": ""BoomBox Malware and Lateral M...",380686dc-684f-4d64-8de1-a7e6e986a66a
3,206,# Network Security Community: Intrusion Preven...,4,8.5,Network Security Community: Intrusion Preventi...,The impact severity rating is high due to the ...,"The community focuses on network security, emp...",[{'explanation': 'Malicious software such as B...,"{\n ""title"": ""Network Security Community: I...",bf7b1b43-d5ef-4bbb-85a9-fb509c38f575
4,207,# Pupy Malware and Cybersecurity Defenses\n\nT...,4,8.5,Pupy Malware and Cybersecurity Defenses,The impact severity rating is high due to the ...,"The community centers around the Pupy malware,...",[{'explanation': 'Pupy is a sophisticated type...,"{\n ""title"": ""Pupy Malware and Cybersecurit...",37b5cd82-e6aa-40ef-9651-5992f501eac0


In [60]:
report_df.iloc[0,1]

"# Windows Operating System and Cyber Threats\n\nThe community centers around the Windows operating system, which is widely used and frequently targeted by various cyber threats and attack techniques. Key entities include security protocols and tools that interact with Windows, highlighting the vulnerabilities and risks associated with this platform.\n\n## Windows as a primary target for cyber threats\n\nWindows is a critical operating system that serves as a primary target for various cyber threats, including malware and sophisticated attack techniques. The operating system's widespread use in both personal and enterprise environments makes it an attractive target for adversaries. Attack techniques such as Adversary-in-the-Middle, DHCP Spoofing, and ARP Cache Poisoning specifically exploit vulnerabilities within Windows, indicating a high level of risk for users. [Data: Entities (112); Relationships (710, 712, 684, 711, 703, +more)]\n\n## Diverse range of malware targeting Windows\n\n

#### Read text units

In [61]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 738


,id,text,n_tokens,document_ids,entity_ids,relationship_ids
0,ea0d9fe4b237628a711bde0bd3bae087,How to mitigate the malicious software P.A.S. ...,42,[000d69d9da85c69d1442caa6397e647c],"[b45241d70f0e43fca764df95b2b81f77, 4119fd06010...",[5154b4a4f3ac43729703c69fccb54633]
1,66e21cf04f235015e12b5e4d52fb8b68,How to mitigate the malicious software Epic? A...,29,[015d8e409fbe48aff60dc1b41386c683],"[d3835bf3dda84ead99deadbeac5d0d7d, 077d2820ae1...","[7cea9903153f43b895c0b23d25bc90a3, b54436ccc23..."
2,978bb2c747ed88cd869d29cb7bd70d53,How to mitigate the malicious software Pupy? E...,33,[0197e2c6f4f05fedc10286ded3417126],"[19a7f254a5d64566ab5cc15472df02de, e7ffaee9d31...","[0057fb2ddc0e4088ae5099b7ffa137da, d67d67cc369..."
3,fdf8849213754403a342f3464c495c13,How to mitigate the malicious software Carberp...,75,[0198c85c7fbeeff76b5bec8fee54d095],"[1fd3fa8bb5a2408790042ab9573779ee, 27f9fbe6ad8...","[7c8c464ed7044a7896adfeb35f58a04d, 09294e8220a..."
4,893b32fffad984aad974026b3597123c,How to mitigate the malicious software Pupy? P...,39,[027e023b88de0f3c25c8913e273c57a6],[19a7f254a5d64566ab5cc15472df02de],[c23761290af24cf29adc1ee8644bdad0]


### Create local search context builder

In [62]:
api_key = os.getenv('OPENAI_API_KEY')
llm_model = "gpt-4o-2024-08-06"
embedding_model = "text-embedding-3-small"

llm = ChatOpenAI(
    api_key=api_key,
    model=llm_model,
    api_type=OpenaiApiType.OpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [63]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    # covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

### Create local search engine

In [64]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 5,
    "top_k_relationships": 5,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 3000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}

In [65]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraph",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

### Run local search on sample queries

In [66]:
query = """
How to mitigate the malicious software Windows Credential Editor?
"""
result = await search_engine.asearch(query)
print(result.response)

## Mitigation Strategies for Windows Credential Editor

Windows Credential Editor (WCE) is a potent malicious software tool that poses significant security risks to Windows systems by facilitating unauthorized access and credential theft. To effectively mitigate the threats posed by WCE, several strategies can be employed.

### Disabling Windows Remote Management (WinRM)

One of the primary methods to mitigate the risk associated with WCE is to disable the Windows Remote Management (WinRM) service. This service, if left enabled, can be exploited by attackers to remotely interact with the system core, thereby increasing the risk of credential theft via remote protocols. By disabling WinRM, you can significantly reduce the potential for remote access to critical systems, thus acting as a barrier against unauthorized intrusions [Data: Sources (567, 230, 670, 61)].

### Implementing Credential Access Protection

Credential Access Protection is another effective mitigation strategy. This ap

In [67]:
print(result.context_data)

{'entities':     id                        entity  \
0  169     WINDOWS CREDENTIAL EDITOR   
1   51              CREDENTIAL GUARD   
2  335         OS CREDENTIAL DUMPING   
3  509              CREDENTIAL THEFT   
4  746   SECURING ACCESS CREDENTIALS   
5  773         CREDENTIAL MANAGEMENT   
6  112                       WINDOWS   
7   64                   CREDENTIALS   
8  207                     CACHEDUMP   
9  282  CREDENTIAL ACCESS PROTECTION   

                                         description number of relationships  \
0  Windows Credential Editor is a malicious softw...                       6   
1  Credential Guard is a robust security feature ...                       9   
2  OS Credential Dumping is a technique employed ...                      19   
3                                                                          1   
4                                                                          1   
5  Credential management involves the processes a...              

In [68]:
result.context_text

"[]\n\n-----Entities-----\nid|entity|description|number of relationships\n169|WINDOWS CREDENTIAL EDITOR|Windows Credential Editor is a malicious software tool designed to manipulate and manage Windows credentials, posing significant security risks to Windows systems. It facilitates unauthorized access by allowing attackers to exploit vulnerabilities in the system for credential theft. This tool is particularly known for its capability to perform password dumping, extracting credentials from Windows systems, and manipulating credentials stored in the Local Security Authority Subsystem Service (LSASS) memory. Additionally, Windows Credential Editor can be utilized for credential theft attempts via remote protocols, further enhancing its potential for malicious use. Overall, it represents a serious threat to the integrity and security of Windows environments.|6\n51|CREDENTIAL GUARD|Credential Guard is a robust security feature designed to safeguard access credentials from being compromise

#### Inspecting the context data used to generate the response

In [72]:
result.context_data["entities"]

,id,entity,description,number of relationships,in_context
0,169,WINDOWS CREDENTIAL EDITOR,Windows Credential Editor is a malicious softw...,6,True
1,51,CREDENTIAL GUARD,Credential Guard is a robust security feature ...,9,True
2,335,OS CREDENTIAL DUMPING,OS Credential Dumping is a technique employed ...,19,True
3,509,CREDENTIAL THEFT,,1,True
4,746,SECURING ACCESS CREDENTIALS,,1,True
5,773,CREDENTIAL MANAGEMENT,Credential management involves the processes a...,2,True
6,112,WINDOWS,Windows is a widely used operating system deve...,55,True
7,64,CREDENTIALS,Credentials are the information used to verify...,5,True
8,207,CACHEDUMP,CACHEDUMP is a publicly-available tool designe...,6,True
9,282,CREDENTIAL ACCESS PROTECTION,Credential Access Protection is a mitigation m...,11,True


In [74]:
result.context_data["relationships"]

KeyError: 'relationships'

In [75]:
df3 = result.context_data["relationships"]

KeyError: 'relationships'

In [76]:
result.context_data["sources"]

,id,text
0,567,How to mitigate the malicious software Windows...
1,230,How to mitigate the malicious software Windows...
2,644,There are 7 mitigation methods that can mitiga...
3,670,How to mitigate the malicious software Windows...
4,61,How to mitigate the malicious software Windows...


In [ ]:
# if "claims" in result.context_data:
#     print(result.context_data["claims"].head())

### Question Generation

This function takes a list of user queries and generates the next candidate questions.

In [ ]:
# question_generator = LocalQuestionGen(
#     llm=llm,
#     context_builder=context_builder,
#     token_encoder=token_encoder,
#     llm_params=llm_params,
#     context_builder_params=local_context_params,
# )
# question_history = [
#     "Tell me about Agent Mercer",
#     "What happens in Dulce military base?",
# ]
# candidate_questions = await question_generator.agenerate(
#     question_history=question_history, context_data=None, question_count=5
# )
# print(candidate_questions.response)

In [ ]:
import networkx as nx
from pyvis.network import Network
import random

# Load the GraphML file
G = nx.read_graphml('/data/yuhui/6/graphrag/alltest/location_dataset/dataset_4_revised/output/20241012-123311/artifacts/merged_graph.graphml')
# Create a Pyvis network
net = Network(notebook=True)

# Convert NetworkX graph to Pyvis network
net.from_nx(G)

# Add colors to nodes
for node in net.nodes:
    node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

# Save and display the network
net.show('knowledge_graph.html')

ModuleNotFoundError: No module named 'pyvis'